# 🌿 Leaf Anomaly Detection — Full Pipeline

**Goal:** Detect diseases in tomato leaves using a trained Autoencoder.

**Method:** Train only on healthy leaves → anything that reconstructs badly = anomaly.

**Run all cells in order:** Runtime → Run all

---

### Pipeline Overview
```
1. Clone repo + install dependencies
2. Mount Drive + download dataset (Kaggle)
3. Organize data into train/test structure
4. Extract DINOv2 features (cached)
5. Train PCA
6. Train Autoencoder
7. Evaluate + visualize results
```

## 🔧 Step 1 — Environment Setup

In [ ]:
# ── Clone repository ─────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/YOUSSEF-ETTABAA/leaf-anomaly-detection'
REPO_NAME = 'leaf-anomaly-detection'

if not os.path.exists(f'/content/{REPO_NAME}'):
    print('Cloning repository...')
    os.system(f'git clone {REPO_URL}')
else:
    print('Repository already exists — pulling latest changes...')
    os.system(f'cd /content/{REPO_NAME} && git pull')

# Move into project directory
os.chdir(f'/content/{REPO_NAME}')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Install dependencies ─────────────────────────────────────
print('Installing dependencies...')
os.system('pip install -r requirements.txt -q')
print('✅ Dependencies installed')

In [ ]:
# ── Verify GPU is available ───────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️  No GPU detected!')
    print('   Go to: Runtime → Change runtime type → GPU')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('🔥 GPU ready')

In [ ]:
# ── Import all project modules ────────────────────────────────
import sys
import numpy as np
import joblib

sys.path.insert(0, '/content/leaf-anomaly-detection')

from src.utils.config   import load_config, get_paths
from src.utils.cache    import load_or_compute
from src.data.prepare_data  import prepare_dataset
from src.features.extractor import load_dino_model, extract_features
from src.features.pca       import fit_pca, apply_pca, save_pca, load_pca
from src.models.autoencoder import load_model, compute_scores
from src.training.train     import train_autoencoder, plot_training_loss
from src.evaluation.metrics import find_threshold, compute_all_metrics, print_metrics
from src.evaluation.plots   import plot_evaluation_dashboard

print('✅ All modules imported successfully')

## 💾 Step 2 — Mount Drive & Setup Paths

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Base path on Google Drive (all large files stored here)
DRIVE_BASE = '/content/drive/MyDrive/leaf-anomaly-detection'

# Load config from repo
cfg   = load_config('/content/leaf-anomaly-detection/config.yaml')
PATHS = get_paths(cfg, DRIVE_BASE)

# Create all directories
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

print('\nProject paths:')
for name, path in PATHS.items():
    print(f'  {name:<18} : {path}')

## 📥 Step 3 — Download & Organize Dataset

In [ ]:
# ── Check if data already exists ─────────────────────────────
from pathlib import Path

train_count = len(list(Path(PATHS['train_healthy']).glob('*.*')))
test_h_count = len(list(Path(PATHS['test_healthy']).glob('*.*')))
test_a_count = len(list(Path(PATHS['test_anomaly']).glob('*.*')))

data_ready = train_count > 0 and test_h_count > 0 and test_a_count > 0

print(f'Train healthy : {train_count} images')
print(f'Test  healthy : {test_h_count} images')
print(f'Test  anomaly : {test_a_count} images')

if data_ready:
    print('\n✅ Data already organized — skipping download')
else:
    print('\n⚠️  Data not found — running Kaggle download...')

In [ ]:
# ── Download from Kaggle (only if data not ready) ─────────────
# Skip this cell if data already exists

if not data_ready:
    # Setup Kaggle credentials
    # Get your API key from: kaggle.com → Settings → API → Create New Token
    KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # ← replace this
    KAGGLE_KEY      = 'YOUR_KAGGLE_KEY'       # ← replace this

    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(f'{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    # Download PlantVillage to Colab local storage (faster than Drive)
    os.system('pip install kaggle -q')
    os.system('kaggle datasets download -d arjuntejaswi/plant-village -p /content/plantvillage --unzip -q')
    print('✅ Dataset downloaded')

    # Organize into our structure
    print('\nOrganizing dataset...')
    counts = prepare_dataset(
        source_path  = '/content/plantvillage/PlantVillage',
        dest_path    = DRIVE_BASE + '/data',
        train_split  = cfg['dataset']['train_split']
    )
else:
    print('Skipping download — data already ready')

## 🔬 Step 4 — Extract DINOv2 Features

In [ ]:
# ── Load DINOv2 ───────────────────────────────────────────────
dino_model, device = load_dino_model(cfg['dino']['model_name'])

In [ ]:
# ── Extract features (cached) ─────────────────────────────────
# First run: takes 10-30 min depending on dataset size
# Next runs: loads from cache instantly

print('Extracting train features...')
train_features = load_or_compute(
    path       = os.path.join(PATHS['cache_features'], 'train_features.npy'),
    compute_fn = lambda: extract_features(
        PATHS['train_healthy'], dino_model, device,
        batch_size = cfg['dino']['batch_size'],
        image_size = cfg['dino']['image_size']
    )
)

print('\nExtracting test healthy features...')
test_h_features = load_or_compute(
    path       = os.path.join(PATHS['cache_features'], 'test_healthy_features.npy'),
    compute_fn = lambda: extract_features(
        PATHS['test_healthy'], dino_model, device,
        batch_size = cfg['dino']['batch_size'],
        image_size = cfg['dino']['image_size']
    )
)

print('\nExtracting test anomaly features...')
test_a_features = load_or_compute(
    path       = os.path.join(PATHS['cache_features'], 'test_anomaly_features.npy'),
    compute_fn = lambda: extract_features(
        PATHS['test_anomaly'], dino_model, device,
        batch_size = cfg['dino']['batch_size'],
        image_size = cfg['dino']['image_size']
    )
)

print(f'\nFeature shapes:')
print(f'  Train    : {train_features.shape}')
print(f'  Test (H) : {test_h_features.shape}')
print(f'  Test (A) : {test_a_features.shape}')
print('✅ Feature extraction complete')

## 📉 Step 5 — PCA Dimensionality Reduction

In [ ]:
# ── Fit or load PCA ───────────────────────────────────────────
pca_scaler_path = os.path.join(PATHS['cache_pca'], 'scaler.pkl')

if os.path.exists(pca_scaler_path):
    print('Loading PCA from cache...')
    scaler, pca = load_pca(PATHS['cache_pca'])
else:
    print('Fitting PCA on training features...')
    _, scaler, pca = fit_pca(
        train_features,
        n_components = cfg['pca']['n_components']
    )
    save_pca(scaler, pca, PATHS['cache_pca'])

# Apply PCA to all splits
# Important: use the SAME scaler+pca fitted on train data
print('\nApplying PCA to all feature sets...')
train_reduced = apply_pca(train_features,   scaler, pca)
test_h_reduced = apply_pca(test_h_features, scaler, pca)
test_a_reduced = apply_pca(test_a_features, scaler, pca)

print(f'\nAfter PCA reduction:')
print(f'  Train    : {train_reduced.shape}')
print(f'  Test (H) : {test_h_reduced.shape}')
print(f'  Test (A) : {test_a_reduced.shape}')
print('✅ PCA complete')

## 🧠 Step 6 — Train Autoencoder

In [ ]:
# ── Train or load autoencoder ─────────────────────────────────
ae_cfg      = cfg['autoencoder']
models_dir  = PATHS['models']
final_path  = os.path.join(models_dir, 'autoencoder_final.pth')

if os.path.exists(final_path):
    print('Loading saved autoencoder...')
    ae_model, device = load_model(
        path       = final_path,
        input_dim  = ae_cfg['input_dim'],
        latent_dim = ae_cfg['latent_dim']
    )
else:
    print('Training autoencoder from scratch...')
    print(f'  Training on {len(train_reduced)} healthy samples')
    ae_model, loss_history = train_autoencoder(
        train_features   = train_reduced,
        save_dir         = models_dir,
        input_dim        = ae_cfg['input_dim'],
        latent_dim       = ae_cfg['latent_dim'],
        epochs           = ae_cfg['epochs'],
        batch_size       = ae_cfg['batch_size'],
        learning_rate    = ae_cfg['learning_rate'],
        checkpoint_every = ae_cfg['checkpoint_every'],
        device           = device
    )

    # Plot training loss
    plot_training_loss(
        loss_history,
        save_path = os.path.join(PATHS['outputs'], 'training_loss.png')
    )

print('✅ Autoencoder ready')

## 📊 Step 7 — Compute Anomaly Scores

In [ ]:
# ── Compute reconstruction error for all test images ──────────
print('Computing anomaly scores...')
h_scores = compute_scores(ae_model, test_h_reduced, device)
a_scores = compute_scores(ae_model, test_a_reduced, device)

print(f'\nScore statistics:')
print(f'  Healthy ({len(h_scores)} images):')
print(f'    mean={h_scores.mean():.4f}  std={h_scores.std():.4f}')
print(f'    min={h_scores.min():.4f}   max={h_scores.max():.4f}')
print(f'  Anomaly ({len(a_scores)} images):')
print(f'    mean={a_scores.mean():.4f}  std={a_scores.std():.4f}')
print(f'    min={a_scores.min():.4f}   max={a_scores.max():.4f}')
print(f'\n  Separation gap : {a_scores.mean() - h_scores.mean():.4f}')

# Find threshold using healthy scores percentile
threshold = find_threshold(h_scores, percentile=cfg['threshold']['percentile'])
print('✅ Scores computed')

## 📈 Step 8 — Evaluation

In [ ]:
# ── Compute all metrics ───────────────────────────────────────
metrics = compute_all_metrics(h_scores, a_scores, threshold)
print_metrics(metrics)

In [ ]:
# ── Detailed classification report ───────────────────────────
from sklearn.metrics import classification_report
print('DETAILED CLASSIFICATION REPORT')
print('='*50)
print(classification_report(
    metrics['y_true'], metrics['y_pred'],
    target_names=['Healthy', 'Anomaly'], digits=4
))

In [ ]:
# ── Full evaluation dashboard ─────────────────────────────────
plot_evaluation_dashboard(
    metrics,
    save_path = os.path.join(PATHS['outputs'], 'evaluation_dashboard.png')
)
print('✅ Dashboard saved')

## 🧪 Step 9 — Test on a Single Image

In [ ]:
import timm
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

def predict_single(image_path, dino_model, ae_model, scaler, pca, threshold, device, image_size=518):
    """
    Predicts whether a single leaf image is healthy or anomalous.

    Args:
        image_path : path to image file
        ...        : loaded models

    Returns:
        verdict (str), score (float)
    """
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    # Extract features
    img = Image.open(image_path).convert('RGB')
    x   = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        pf = dino_model.forward_features(x)[:, 1:, :].squeeze(0).cpu().numpy()

    gs      = int(np.sqrt(pf.shape[0]))
    feat    = pca.transform(scaler.transform(pf[:gs*gs].mean(axis=0, keepdims=True)))
    score   = float(compute_scores(ae_model, feat, device)[0])
    pred    = 'ANOMALY' if score >= threshold else 'HEALTHY'
    icon    = '🚨' if pred == 'ANOMALY' else '✅'
    color   = '#c62828' if pred == 'ANOMALY' else '#2e7d32'

    # Plot result
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.resize((300, 300)))
    axes[0].set_title('Input Leaf', fontweight='bold')
    axes[0].axis('off')

    axes[1].axis('off')
    axes[1].text(0.5, 0.70, f'{icon}  {pred}', ha='center', va='center',
                 fontsize=30, fontweight='bold', color=color, transform=axes[1].transAxes)
    axes[1].text(0.5, 0.50, f'Score     : {score:.6f}', ha='center', va='center',
                 fontsize=13, color='#333333', transform=axes[1].transAxes, fontfamily='monospace')
    axes[1].text(0.5, 0.37, f'Threshold : {threshold:.6f}', ha='center', va='center',
                 fontsize=13, color='#888888', transform=axes[1].transAxes, fontfamily='monospace')
    plt.suptitle('Single Leaf Prediction', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Verdict   : {icon} {pred}')
    print(f'Score     : {score:.6f}')
    print(f'Threshold : {threshold:.6f}')
    return pred, score


# ── Test on a sample image ────────────────────────────────────
# Change this path to test your own image
sample = str(list(Path(PATHS['test_anomaly']).glob('*.JPG'))[0])
predict_single(sample, dino_model, ae_model, scaler, pca, threshold, device)